# DermaMNIST — correctness memory (full pipeline)

Classifier training is **unchanged**. Only the associative side-channel memory uses a correctness target.

## Implementation equations (audit)

$$k_t = \frac{h_t}{\|h_t\|_2 + \epsilon}$$

$$c_t = \mathbf{1}[\hat y_t = y_t], \qquad u_t = c_t P k_t \in \mathbb{R}^{7}$$

$$M^{(j)} \in \mathbb{R}^{7 \times d_k}, \quad z_t^{(j)} = M^{(j)} k_t \in \mathbb{R}^{7}$$

$$M_{t+1}^{(j)} = M_t^{(j)} + \eta_j (c_t P k_t - M_t^{(j)} k_t) k_t^\top$$

Deploy: $z_{\mathrm{combined}}=[z_1,\ldots,z_4]\in\mathbb{R}^{28}$, logistic probe $q(x)=\sigma(w^\top z_{\mathrm{combined}}+b)$, $s_{\mathrm{fail}}=1-q(x)$.

Combines `dermamnist_full_validation.ipynb` (train + deploy) and `memory_vector_vs_magnitude_probe.ipynb` (probe eval).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

TASK = "dermamnist"
CORRECTNESS_Z_DIM = 7  # z_j in R^7; z_combined in R^{4*7}=28

# Full probe protocol uses (42, 123, 456); use a subset for quick iteration.
SEEDS = (42, 123, 456)

# Part I — subsample count from the train split (None = use all train images).
TRAIN_SAMPLE_SIZE = 5000

# Part III — subsample count from cached full test features for probe metrics.
TEST_SAMPLE_SIZE = 2000

# True → wipe per-seed train/deploy caches and rerun Parts I–II. False → skip if artifacts exist.
RETRAIN = False

OUTPUT_DIR = REPO_ROOT / "research" / "correctness_memory_fft" / "artifacts"
FEATURE_CACHE = REPO_ROOT / "research" / "correctness_memory_fft" / "probe_experiments" / "correctness_feature_cache"
HT_CACHE = REPO_ROOT / "research" / "correctness_memory_fft" / "probe_experiments" / "ht_probe_cache"

print(REPO_ROOT)
print(
    f"SEEDS={SEEDS}  CORRECTNESS_Z_DIM={CORRECTNESS_Z_DIM}  "
    f"TRAIN_SAMPLE_SIZE={TRAIN_SAMPLE_SIZE}  "
    f"TEST_SAMPLE_SIZE={TEST_SAMPLE_SIZE}  RETRAIN={RETRAIN}"
)

c:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection
SEEDS=(42, 123, 456)  CORRECTNESS_Z_DIM=7  TRAIN_SAMPLE_SIZE=5000  TEST_SAMPLE_SIZE=2000  RETRAIN=True


## Part I — Train (classifier + correctness memory)

In [6]:
import pickle
import shutil
from dataclasses import replace

import numpy as np
from optimizer.associative_memory import AssociativeMemoryConfig
from research.common.clinical_datasets import ClinicalDatasetConfig, load_clinical_bundle
from research.common.clinical_training import ClinicalTrainingConfig, checkpoint_path, save_checkpoint
from research.common.correctness_clinical_training import train_one_seed_correctness
from research.common.correctness_memory_io import (
    bootstrap_classifier_checkpoint_if_missing,
    correctness_artifacts_exist,
    correctness_checkpoint_bundle_path,
    correctness_memory_exists,
    correctness_memory_path,
    recover_correctness_checkpoint_bundle,
)

RESIDUAL_OUTPUT_DIR = REPO_ROOT / "research" / "associative_memory_fft" / "artifacts"
DATA_DIR = REPO_ROOT / "data" / "clinical"
bundle = load_clinical_bundle(ClinicalDatasetConfig(task=TASK, data_dir=DATA_DIR))
train_cfg = ClinicalTrainingConfig(
    seeds=SEEDS,
    output_dir=OUTPUT_DIR,
    verbose=1,
    associative_memory=AssociativeMemoryConfig(correctness_z_dim=CORRECTNESS_Z_DIM),
)


def subset_train_bundle(bundle, n: int | None, seed: int):
    if n is None or len(bundle.x_train) <= n:
        return bundle
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(bundle.x_train), size=n, replace=False))
    train_ids = bundle.sample_ids["train"][idx]
    return replace(
        bundle,
        x_train=bundle.x_train[idx],
        y_train=bundle.y_train[idx],
        sample_ids={**bundle.sample_ids, "train": train_ids},
    )


train_bundle = subset_train_bundle(bundle, TRAIN_SAMPLE_SIZE, seed=min(SEEDS))
print(f"training on {len(train_bundle.x_train)} samples (TRAIN_SAMPLE_SIZE={TRAIN_SAMPLE_SIZE})")

for seed in SEEDS:
    ckpt = checkpoint_path(OUTPUT_DIR, TASK, seed)

    if RETRAIN:
        for path in (
            ckpt,
            correctness_memory_path(OUTPUT_DIR, TASK, seed),
            correctness_checkpoint_bundle_path(OUTPUT_DIR, TASK, seed),
        ):
            if path.exists():
                path.unlink()
        for cache_dir in (FEATURE_CACHE / f"seed{seed}", HT_CACHE / f"seed{seed}"):
            if cache_dir.exists():
                shutil.rmtree(cache_dir)
        print(f"seed {seed}: cleared artifacts (RETRAIN=True)")
    elif correctness_memory_exists(OUTPUT_DIR, TASK, seed):
        recover_correctness_checkpoint_bundle(OUTPUT_DIR, TASK, seed)
        bootstrap_classifier_checkpoint_if_missing(OUTPUT_DIR, RESIDUAL_OUTPUT_DIR, TASK, seed)
        if correctness_artifacts_exist(OUTPUT_DIR, TASK, seed) and ckpt.exists():
            print(f"seed {seed}: recovered memory + classifier (no retrain)")
            continue
    elif ckpt.exists() and correctness_artifacts_exist(OUTPUT_DIR, TASK, seed):
        print(f"seed {seed}: skip (artifacts exist)")
        continue

    params, opt_state, _, _, summary = train_one_seed_correctness(
        train_bundle, seed=seed, cfg=train_cfg
    )
    save_checkpoint(ckpt, params=params, opt_state=opt_state, task=TASK, seed=seed, cfg=train_cfg, summary=summary)
    print(f"seed {seed}: test_acc={summary['test_acc']:.4f}")

training on 5000 samples (TRAIN_SAMPLE_SIZE=5000)
seed 42: cleared artifacts (RETRAIN=True)
  [dermamnist seed=42] correctness-memory training: 5000 samples, 8 epochs, ~79 batches/epoch
  [dermamnist seed=42] epoch 1/8: train_loss=0.9676 cal_acc=0.677 test_acc=0.672 (625.8s)
  [dermamnist seed=42] epoch 2/8: train_loss=0.8760 cal_acc=0.669 test_acc=0.670 (568.5s)
  [dermamnist seed=42] epoch 3/8: train_loss=0.8376 cal_acc=0.682 test_acc=0.687 (572.6s)
  [dermamnist seed=42] epoch 4/8: train_loss=0.8257 cal_acc=0.699 test_acc=0.696 (573.5s)
  [dermamnist seed=42] epoch 5/8: train_loss=0.7970 cal_acc=0.682 test_acc=0.705 (581.1s)
  [dermamnist seed=42] epoch 6/8: train_loss=0.7573 cal_acc=0.717 test_acc=0.709 (584.3s)
  [dermamnist seed=42] epoch 7/8: train_loss=0.7373 cal_acc=0.718 test_acc=0.721 (593.3s)
  [dermamnist seed=42] epoch 8/8: train_loss=0.7103 cal_acc=0.702 test_acc=0.712 (608.7s)
seed 42: test_acc=0.7117
seed 123: cleared artifacts (RETRAIN=True)
  [dermamnist seed=123] co

## Part II — Deploy features ($s_j$, $f_j=1-s_j$)

In [7]:
from research.common.correctness_deployment_pipeline import run_correctness_deployment_on_split, run_correctness_sanity_checks
from research.common.correctness_memory_io import load_correctness_artifacts

for seed in SEEDS:
    seed_dir = FEATURE_CACHE / f"seed{seed}"
    cal_path = seed_dir / "cal_features.csv"
    test_path = seed_dir / "test_features_full.csv"
    if not RETRAIN and cal_path.exists() and test_path.exists():
        print(f"seed {seed}: feature cache exists")
        continue
    seed_dir.mkdir(parents=True, exist_ok=True)
    with checkpoint_path(OUTPUT_DIR, TASK, seed).open("rb") as f:
        params = pickle.load(f)["params"]
    mem, _, _ = load_correctness_artifacts(OUTPUT_DIR, TASK, seed)
    sanity = run_correctness_sanity_checks(params, bundle.x_cal[:8], mem, num_classes=bundle.num_classes, labels=bundle.y_cal[:8])
    print(f"seed {seed} sanity:", sanity["passed"], sanity["checks"])
    for split, x, y, ids in [
        ("cal", bundle.x_cal, bundle.y_cal, bundle.sample_ids["cal"]),
        ("test", bundle.x_test, bundle.y_test, bundle.sample_ids["test"]),
    ]:
        _, df = run_correctness_deployment_on_split(params, x, y, ids, mem, num_classes=bundle.num_classes)
        out = cal_path if split == "cal" else test_path
        df.to_csv(out, index=False)
    print(f"seed {seed}: deployed")

seed 42 sanity: True {'key_dim_positive': True, 'key_unit_norm': True, 'response_dim_L1': True, 'manual_response_L1': True, 'response_dim_L2': True, 'manual_response_L2': True, 'response_dim_L3': True, 'manual_response_L3': True, 'response_dim_L4': True, 'manual_response_L4': True, 'matrix_shapes_dv_x_dk': True, 'c_t_binary_sample_0': True, 'c_t_binary_sample_1': True, 'c_t_binary_sample_2': True, 'c_t_binary_sample_3': True, 'c_t_binary_sample_4': True, 'all_passed': True}
seed 42: deployed
seed 123 sanity: True {'key_dim_positive': True, 'key_unit_norm': True, 'response_dim_L1': True, 'manual_response_L1': True, 'response_dim_L2': True, 'manual_response_L2': True, 'response_dim_L3': True, 'manual_response_L3': True, 'response_dim_L4': True, 'manual_response_L4': True, 'matrix_shapes_dv_x_dk': True, 'c_t_binary_sample_0': True, 'c_t_binary_sample_1': True, 'c_t_binary_sample_2': True, 'c_t_binary_sample_3': True, 'c_t_binary_sample_4': True, 'all_passed': True}
seed 123: deployed
seed

## Part III — Probe comparison

Same leakage-safe protocol as the residual-memory probe notebook:

| Model | Description |
|-------|-------------|
| **H** | Normalized entropy (label-free) |
| **h_T probe** | Cal-fit logistic on 512-d penultimate features |
| **z_combined** | Cal-fit logistic on $z_{\mathrm{combined}}\in\mathbb{R}^{28}$ |
| **z1..z4 ablation** | Cumulative $z_1,\ldots,z_j$ blocks (7 dims each) |

`run_analysis(seeds=SEEDS)` extracts h_T once per seed (cached under `ht_probe_cache/`).

In [8]:
from research.correctness_memory_fft.probe_experiments.correctness_probe_analysis import (
    ablation_summary_table,
    comparison_summary_table,
    run_analysis,
)

summary = run_analysis(seeds=SEEDS, test_sample_size=TEST_SAMPLE_SIZE)
p3 = summary["phase3_summary"]

print(f"z_combined dim={summary['z_combined_dim']}  z_dim={summary['z_dim']}")
print(f"test_sample_size={summary['test_sample_size']}")
print(f"z_combined AUROC={p3['mean_auroc_z_combined']:.4f}")
print(f"entropy H AUROC={p3['mean_auroc_H']:.4f}")
print(f"h_T probe AUROC={p3['mean_auroc_h_T_probe']:.4f}")
print(f"z - H = {p3['mean_delta_z_minus_H']:+.4f}")
print(f"z - h_T = {p3['mean_delta_z_minus_h_T']:+.4f}")

    extracted h_T seed=42 cal n=1602 in 41.5s
    extracted h_T seed=42 test n=2005 in 52.1s
    extracted h_T seed=123 cal n=1602 in 41.4s
    extracted h_T seed=123 test n=2005 in 54.0s
    extracted h_T seed=456 cal n=1602 in 42.0s
    extracted h_T seed=456 test n=2005 in 54.0s
z_HD dim=28  z_dim=7
test_sample_size=2000
z_combined AUROC=0.7531
entropy H AUROC=0.6645
h_T probe AUROC=0.7067
z - H = +0.0886
z - h_T = +0.0464


In [9]:
import pandas as pd

comparison_summary_table(summary)

,mean_auroc,mean_auprc
model,,
entropy_H,0.664466,0.496894
h_T_probe,0.706658,0.556842
z_combined,0.753084,0.606137


In [10]:
phase3 = pd.DataFrame(summary["phase3_per_seed"])
phase3.pivot(index="seed", columns="model", values="auroc").round(4)

model,entropy_H,h_T_probe,z_combined
seed,,,
42,0.6507,0.7329,0.7648
123,0.6854,0.6973,0.7501
456,0.6573,0.6898,0.7444


In [11]:
import matplotlib.pyplot as plt

ab = pd.DataFrame(summary["ablation_per_seed"])
ab_mean = ab.groupby("max_level")["auroc"].mean()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

models = comparison_summary_table(summary).reset_index()
colors = {"entropy_H": "#DD8452", "h_T_probe": "#55A868", "z_combined": "#4C72B0"}
axes[0].bar(
    models["model"],
    models["mean_auroc"],
    color=[colors.get(m, "#999999") for m in models["model"]],
)
axes[0].set_ylabel("Test AUROC (mean over seeds)")
axes[0].set_title("z_combined vs H vs h_T")
axes[0].tick_params(axis="x", rotation=20)

axes[1].plot(ab_mean.index, ab_mean.values, "-o", color="#4C72B0")
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_xlabel("Cumulative levels")
axes[1].set_ylabel("Test AUROC (mean over seeds)")
axes[1].set_title("Cumulative ablation z1..z4 (7-d blocks)")

fig.tight_layout()
plt.show()

ablation_summary_table(summary)

C:\Users\Sounak Sinha\AppData\Local\Temp\ipykernel_18996\3902971945.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mean_auroc,mean_auprc
levels,,
z1..z1,0.728134,0.553530
z1..z2,0.753049,0.604467
z1..z3,0.752908,0.605345
z1..z4,0.753084,0.606137


## Optional — run full script

Equivalent to all cells above:

In [ ]:
# %run notebooks/_execute_correctness_derma_pipeline.py